<a href="https://colab.research.google.com/github/astroelaa/customer_support_agent/blob/main/cu_support_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Customer support AI agent with tool calling,
human handoff &  Gradio UI (CrewAI + Gemini).

imports

In [ ]:
%pip install -q "crewai[tools]"

llm , the brain of the agent

In [ ]:
import os
import json
from google.colab import userdata
from crewai import LLM
from crewai.tools import tool

try:
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
except Exception:
    import getpass
    print("GEMINI_API_KEY secret not found in Colab Secrets.")
    os.environ["GEMINI_API_KEY"] = getpass.getpass("Please paste your Gemini API Key here and press Enter: ")

llm = LLM(model="gemini/gemini-3.1-flash-lite")

orders

In [ ]:
ORDERS = {
    "1001": {"customer": "Sara", "item": "Wireless Headphones", "total": 1200, "status": "shipped", "tracking": "TRK-777", "refunded": False},
    "1002": {"customer": "Omar", "item": "Phone Case", "total": 150, "status": "processing", "tracking": None, "refunded": False},
    "1003": {"customer": "Nour", "item": "Laptop Stand", "total": 800, "status": "delivered", "tracking": "TRK-555", "refunded": False},
}

shipping data , linked by orders, tickets are empty to create a ticket later.

In [ ]:
SHIPMENTS = {
    "TRK-777": {"carrier": "Bosta", "location": "Cairo hub", "delayed": True, "reason": "high volume", "eta": "2026-10-12"},
    "TRK-555": {"carrier": "Aramex", "location": "Delivered", "delayed": False, "reason": None, "eta": None},
}

TICKETS = []

@tool makes a normal fun, can be called later by the agent
the tools return Json strings bc the llms reads text
the agent wont see the code. it only sees the docstring & the name
check shipping here needs a tracking num, only get get order . so the agent calls get order ,
the is the tool chaining & it needsno extra code.


In [ ]:
@tool("get_order")
def get_order(order_id: str) -> str:
    """Look up an order by its ID. Returns customer, item, total, status and tracking number."""
    order = ORDERS.get(order_id)
    if not order:
        return json.dumps({"error": f"Order {order_id} not found"})
    return json.dumps({"order_id": order_id, **order})


@tool("check_shipping")
def check_shipping(tracking_number: str) -> str:
    """Check shipment status using a tracking number. Returns carrier, location, delay flag, reason and ETA."""
    shipment = SHIPMENTS.get(tracking_number)
    if not shipment:
        return json.dumps({"error": f"Tracking number {tracking_number} not found"})
    return json.dumps({"tracking_number": tracking_number, **shipment})

the business rule lives inside the tool itself , not js the prompt , so the llm can ignore the insturctions , bt t cannot bypass the code, the tool refures , the error text goes back to the agent so it reads it and change the  plan

In [ ]:
@tool("cancel_order")
def cancel_order(order_id: str) -> str:
    """Cancel an order. Only works if the order status is 'processing'. Shipped or delivered orders cannot be cancelled."""
    order = ORDERS.get(order_id)
    if not order:
        return json.dumps({"error": f"Order {order_id} not found"})
    if order["status"] != "processing":
        return json.dumps({"error": f"Cannot cancel, order status is {order['status']}"})
    order["status"] = "cancelled"
    return json.dumps({"success": True, "order_id": order_id, "status": "cancelled"})

order exists, status allows it, not already refunded. The "already refunded" guard matters because without it the agent could refund the same order twice. Note that cancelling then refunding is a natural chain for the agent.

In [ ]:
REFUND_LIMIT = 500

@tool("refund_order")
def refund_order(order_id: str) -> str:
    """Refund an order. Only works if the order is 'delivered' or 'cancelled' and not already refunded. Refunds above 500 need human approval, so create a ticket instead."""
    order = ORDERS.get(order_id)
    if not order:
        return json.dumps({"error": f"Order {order_id} not found"})
    if order["status"] not in ("delivered", "cancelled"):
        return json.dumps({"error": f"Cannot refund, order status is {order['status']}"})
    if order["refunded"]:
        return json.dumps({"error": "Order already refunded"})
    if order["total"] > REFUND_LIMIT:
        return json.dumps({"error": f"Refund of {order['total']} is above the limit of {REFUND_LIMIT}, a human must approve it"})
    order["refunded"] = True
    return json.dumps({"success": True, "order_id": order_id, "amount": order["total"]})

 The agent also writes the issue and priority itself, so it summarizes the problem for the human who picks it up.

In [ ]:
@tool("create_ticket")
def create_ticket(order_id: str, issue: str, priority: str) -> str:
    """Create a ticket for a human support agent. Use when the problem cannot be solved with the other tools or the customer is very upset. Priority must be low, medium or high."""
    ticket_id = f"T-{len(TICKETS) + 1:03d}"
    TICKETS.append({"ticket_id": ticket_id, "order_id": order_id, "issue": issue, "priority": priority})
    return json.dumps({"success": True, "ticket_id": ticket_id})

In [ ]:
tools = [get_order, check_shipping, cancel_order, refund_order, create_ticket]

The role, goal, & backstory shape how the agent behaves. The tools list is what it can act with. The rules in the backstory are guidance, while the hard rules stay in the tool code. verbose=True prints the agent's thinking &  tool calls, which is how will see &understand its decisions.

In [ ]:
from crewai import Agent

support_agent = Agent(
    role="Customer Support Agent",
    goal="Resolve customer problems about orders using the available tools, or hand off to a human when needed",
    backstory=(
        "You are a polite and efficient support agent. "
        "Always look up the order before taking any action. "
        "Never guess order details, use the tools. "
        "If a tool returns an error, explain it to the customer and try another option. "
        "Create a ticket only if the customer is upset, asks for a human, or the tools cannot solve the problem. "
        "A delayed order with a known ETA does not need a ticket. "
        "Reply to the customer in a short, friendly message."
        "Only refund or cancel when the customer asks for it. "
        "Only state facts returned by the tools, never invent currencies, payment methods or timelines. "

    ),
    tools=tools,
    llm=llm,
    verbose=True,
)



The task says what to do now. {message} is a placeholder that gets filled in each time run the crew. expected_output tells the agent what a finished answer looks like.

In [ ]:
from crewai import Task

support_task = Task(
    description=(
        "A customer sent this message: {message}\n"
        "Understand the problem, use the right tools, decide whether to resolve it or create a ticket, "
        "and write the final reply to the customer."
    ),
    expected_output="A short, friendly reply to the customer that explains what was done or what happens next.",
    agent=support_agent,
)

The Crew runs the task with the agent. run_agent wraps it in one function, so a demo cell is one line, & the Gradio UI later will call this same function.

In [ ]:
import asyncio
from crewai import Crew

crew = Crew(agents=[support_agent], tasks=[support_task])

async def run_agent(message, retries=5):
    for attempt in range(retries):
        try:
            result = await crew.kickoff_async(inputs={"message": message})
            return str(result)
        except Exception:
            if attempt == retries - 1:
                return "Sorry, our system is busy right now. A human agent will contact you shortly."
            await asyncio.sleep(10 * (attempt + 1))



> Demo case 1  , the agent loop , get order & then check shipping , wrote a reply from its descions



In [ ]:
print(await run_agent("My order 1001 is late and I don't know when it will arrive"))

Demo case 2 , Order 1002 is processing, so expect a chain of get_order, cancel_order, then refund_order. The refund only works bc the cancel changed the status to cancelled, so the tools depend on each other's results.

In [ ]:
print(await run_agent("I want to cancel order 1002 and get my money back"))

In [ ]:
print(ORDERS["1002"])

Demo case 3. Order 1001 is shipped, so cancel_order returns an error what the agent does with  it should explain that cancelling is no longer possible & offer another option, such as a ticket or a return after delivery. This is the error recovery from the backstory in action

In [ ]:
print(await run_agent("Cancel my order 1001 right now"))

Demo case 4. Expect get_order, then create_ticket with priority high,&  reply that apologizes & gives the ticket ID. The agent chooses this bc the docstring says to create a ticket when the customer is very upset.

In [ ]:
print(await run_agent("This is the third time my order 1003 has problems, the stand arrived broken and I am very angry, I want a manager"))

This shows the ticket the agent wrote, including the issue text & priority it chose itself.

In [ ]:
print(TICKETS)

Gradio passes the chat message, and run_agent does all the work. The history argument is unused for now. It becomes useful when we add memory.

In [ ]:
import gradio as gr

async def respond(message, history):
    return await run_agent(message)

gr.ChatInterface(respond, title="Customer Support Agent").launch()